# Lab 05 — First Agent with the OpenAI Agents SDK

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-05-first-agent-with-the-openai-agents-sdk/lab-05-first-agent-with-the-openai-agents-sdk.ipynb)

**Topic:** 3 — Multi-Agent System Development with OpenAI Agents SDK

**Objective:** Design an agent with the OpenAI Agents SDK using structured outputs and function tools

Rebuild the hand-rolled agent from Lab 01 on the OpenAI Agents SDK and see how much the framework removes.

Full step-by-step instructions are in the Learner Guide.


**Note the naming:** the PyPI package is `openai-agents` but you import from `agents`. This trips people up constantly.


In [ ]:
!pip install -q openai-agents pydantic python-dotenv requests


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Create an Agent and run it with Runner.run_sync

Compare this against the roughly 90 lines of loop code you wrote in Lab 01 — the SDK's `Runner` handles the entire reason-act-observe cycle, the turn cap and the tool-result plumbing.


In [ ]:
from agents import Agent, Runner

MODEL = "gpt-4o-mini"

first_agent = Agent(
    name="Assistant",
    instructions=(
        "You are a concise research assistant. Use the tools available to you "
        "rather than guessing, and state clearly when you do not know something."
    ),
    model=MODEL,
)

result = Runner.run_sync(first_agent, "What are three benefits of unit testing?")
print(result.final_output)


`Runner.run_sync` is the blocking convenience wrapper. In an async application — including the Streamlit app in Lab 07 — use `await Runner.run(agent, "...")` instead. There is also `Runner.run_streamed(...)` for token-by-token output.


## 2. Define a Pydantic BaseModel as the agent's output_type

Prose is hard for code to consume. An `output_type` forces the model to return validated, typed data. The `Field(description=...)` text is sent to the model, so it is worth writing properly.


In [ ]:
from pydantic import BaseModel, Field


class ResearchBrief(BaseModel):
    """The structured answer shape the agent must return."""

    topic: str = Field(description="The subject of the question, in a few words.")
    summary: str = Field(description="A two-sentence answer.")
    key_points: list[str] = Field(description="Between three and five key points.")
    confidence: float = Field(
        ge=0.0, le=1.0, description="Confidence in the answer, from 0 to 1."
    )


## 3. Decorate a Python function with @function_tool

The docstring and type hints become the schema the model sees — you no longer hand-write the JSON you wrote in Lab 01.


In [ ]:
from agents import function_tool


@function_tool
def calculate(expression: str) -> str:
    """Evaluate an arithmetic expression such as '0.15 * 2400'.

    Use this for any arithmetic instead of computing the answer yourself.

    Args:
        expression: An arithmetic expression using digits and + - * / ( ) only.
    """
    allowed = set("0123456789.+-*/() ")
    if not set(expression) <= allowed:
        return "Error: expression contains unsupported characters."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"Error: could not evaluate '{expression}': {exc}"


## 4. Add a second tool that calls a real public API

So the agent works with live data rather than the stub from Lab 01. Open-Meteo needs no key.


In [ ]:
import requests


@function_tool
def get_weather(latitude: float, longitude: float) -> str:
    """Get the current temperature and wind speed for a location.

    Use this whenever the user asks about current weather conditions.

    Args:
        latitude: Latitude in decimal degrees, e.g. 1.29 for Singapore.
        longitude: Longitude in decimal degrees, e.g. 103.85 for Singapore.
    """
    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m",
            },
            timeout=10,
        )
        response.raise_for_status()
        current = response.json()["current"]
    except requests.RequestException as exc:
        return f"Error: weather service unavailable ({exc})."
    except (KeyError, ValueError) as exc:
        return f"Error: unexpected response from weather service ({exc})."

    return (
        f"Temperature {current['temperature_2m']}degC, "
        f"wind {current['wind_speed_10m']} km/h."
    )


## 5. Handle the failure path

A tool that raises should not crash the run. Two layers of defence, and you want both: return an error string for expected failures (as `get_weather` does), and supply a `failure_error_function` for exceptions you did not anticipate.


In [ ]:
def tool_error(context, error: Exception) -> str:
    """Convert an unhandled tool exception into a message the model can read."""
    return f"The tool failed: {error}. Tell the user this data is unavailable."


@function_tool(failure_error_function=tool_error)
def lookup_population(city: str) -> str:
    """Get the population of a city. Raises if the city is unknown.

    Args:
        city: The city name.
    """
    populations = {"singapore": 5_920_000, "london": 8_900_000}
    return str(populations[city.strip().lower()])  # KeyError on unknown city


## 6. Assemble the agent with tools and the output type


In [ ]:
agent = Agent(
    name="Research Assistant",
    instructions=(
        "You are a concise research assistant. Use the tools available to you "
        "rather than guessing. Always populate every field of the output."
    ),
    model=MODEL,
    tools=[calculate, get_weather, lookup_population],
    output_type=ResearchBrief,
)
print(agent.name, "ready with", len(agent.tools), "tools")


## 7. Run it and inspect the result

`result.final_output` is now a real `ResearchBrief` instance, so `brief.confidence` is a `float` you can compare against a threshold. The result object also shows which tools ran and in what order. Runs are traced automatically at <https://platform.openai.com/traces>.


In [ ]:
result = Runner.run_sync(
    agent,
    "What is the current weather in Singapore (1.29, 103.85), "
    "and what is 15% of 2400?",
)

brief = result.final_output
print(f"Topic:      {brief.topic}")
print(f"Summary:    {brief.summary}")
print(f"Confidence: {brief.confidence:.2f}")
for point in brief.key_points:
    print(f"  - {point}")

print(f"\nisinstance(result.final_output, ResearchBrief): "
      f"{isinstance(result.final_output, ResearchBrief)}")

print("\n--- Run trace ---")
for item in result.new_items:
    print(f"  {type(item).__name__}")

print(f"\nLast agent: {result.last_agent.name}")


## 8. Test the failure path deliberately

The run should complete and report the data is unavailable, rather than terminating with a `KeyError`.


In [ ]:
failure = Runner.run_sync(agent, "What is the population of Atlantis?")
print(failure.final_output.summary)
